# CI integration test: save/restore

Mirrors `examples/mneflow_save_restore.ipynb`: reloads the tfrecords +
trained model that `basic_example_ci.ipynb` wrote to `MNEFLOW_DATA_PATH`,
continues training a few epochs with `collect_patterns=True`, then exercises
prediction and the pattern-interpretation plots. Not a tutorial -- see the
notebook in `examples/` for that. Must run after `basic_example_ci.ipynb`.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import mne
mne.set_log_level(verbose='CRITICAL')

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import mneflow
print(mneflow.__version__)

In [ ]:
n_epochs = int(os.environ.get('MNEFLOW_N_EPOCHS', '3'))
path = os.environ.get('MNEFLOW_DATA_PATH', '/tmp/mneflow_ci/')
data_id = 'mne_sample_multimodal'

In [ ]:
meta = mneflow.utils.load_meta(path, data_id)
model = meta.restore_model()

In [ ]:
# collect_patterns=True saves the spatial/temporal patterns learned in this
# training run so they can be exercised by the plotting cells below.
model.train(n_epochs=n_epochs, collect_patterns=True)

In [ ]:
test_loss, test_acc = model.evaluate(meta.data['test_paths'])
print("Test set: Loss = {:.4f} Accuracy = {:.4f}".format(test_loss, test_acc))

In [ ]:
X, y = [row for row in model.dataset.val.take(1)][0]
y_pred = model.predict_sample(X[0])
print("Predicted: {}, Ground truth {}".format(y_pred[0], np.argmax(y[0])))

In [ ]:
model.meta.plot_spatial_patterns('weight', sensor_layout='Vectorview-grad')

In [ ]:
model.meta.plot_spectra(method='weight', log=False, freqs_lim=(1, 45))

In [ ]:
model.meta.plot_timecourses(freqs_lim=(1, 45), method='weight', average_over=None)

In [ ]:
model.meta.explore_components(sorting='weight', sensor_layout='Vectorview-grad', diff=False, n_cols=1)